In [164]:
import json
import pandas as pd
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer,PorterStemmer
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Prudhvi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [165]:
with open("data/processed/chunk.json","r",encoding="utf-8") as f:
    chunks =json.load(f)

In [166]:
print(len(chunks))

260


In [167]:
en_stopwords=stopwords.words("english")
ps = PorterStemmer()

In [168]:
test=chunks[0]["text"]

In [169]:
test=test.lower()

In [170]:
def tokenize(text):
    text=text.lower()
    text=text.replace("u.s.","us")
    text = re.sub(r"[/\-\u2011\u2013]", " ", text)
    text_token=word_tokenize(text.lower())
    text_token_punt = [re.sub(r"[^\w\s]", "", word) for word in text_token]
    text_token_stopword = [word for word in text_token_punt
                           if word and word not in en_stopwords]

    text_token_stem = [ps.stem(word) for word in text_token_stopword]
    return text_token_stem
    

In [171]:
print(tokenize(test))

['feder', 'reserv', 'commit', 'use', 'full', 'rang', 'tool', 'support', 'us', 'economi', 'challeng', 'time', 'therebi', 'promot', 'maximum', 'employ', 'price', 'stabil', 'goal']


In [172]:
print(tokenize("raise the target range for the federal funds rate to 5-1/4 to 5-1/2 percent longer term"))

['rais', 'target', 'rang', 'feder', 'fund', 'rate', '5', '1', '4', '5', '1', '2', 'percent', 'longer', 'term']


In [173]:
tokenized_chunks=[tokenize(chunk["text"]) for chunk in chunks]

In [174]:
bm25 = BM25Okapi(tokenized_chunks)

In [175]:
print(bm25.corpus_size)
print(bm25.avgdl)    
print(bm25.doc_len[:5]) 

260
41.87692307692308
[19, 53, 24, 85, 48]


In [176]:
def search(query, k=5):
    query_token=tokenize(query)
    scores=bm25.get_scores(query_token)
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    for i in top:
        print(round(scores[i], 2), chunks[i]["chunk_id"], chunks[i]["text"][:100])
    return  [(chunks[i]["chunk_id"], float(scores[i])) for i in top]

In [177]:
search("balance sheet reduction")

6.25 2022-11-02_p3_c2 In addition, the Committee will continue reducing its holdings of Treasury securities and agency deb
6.25 2022-12-14_p3_c2 In addition, the Committee will continue reducing its holdings of Treasury securities and agency deb
5.39 2025-10-29_p3_c1 In support of its goals and in light of the shift in the balance of risks, the Committee decided to 
4.5 2022-09-21_p3_c1 The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the lo
4.48 2021-11-03_p3_c1 The path of the economy continues to depend on the course of the virus. Progress on vaccinations and


[('2022-11-02_p3_c2', 6.246367628121575),
 ('2022-12-14_p3_c2', 6.246367628121575),
 ('2025-10-29_p3_c1', 5.388284977086606),
 ('2022-09-21_p3_c1', 4.495809938428388),
 ('2021-11-03_p3_c1', 4.479965602369564)]

In [178]:
search("inflation transitory")

4.71 2021-07-28_p2_c1 With progress on vaccinations and strong policy support, indicators of economic activity and employm
4.47 2021-09-22_p2_c1 With progress on vaccinations and strong policy support, indicators of economic activity and employm
4.38 2021-06-16_p2_c1 Progress on vaccinations has reduced the spread of COVID-19 in the United States. Amid this progress
4.13 2021-04-28_p2_c1 The COVID-19 pandemic is causing tremendous human and economic hardship across the United States and
3.91 2021-11-03_p2_c1 With progress on vaccinations and strong policy support, indicators of economic activity and employm


[('2021-07-28_p2_c1', 4.714712576599406),
 ('2021-09-22_p2_c1', 4.472143550240558),
 ('2021-06-16_p2_c1', 4.3819638261052045),
 ('2021-04-28_p2_c1', 4.13200131831041),
 ('2021-11-03_p2_c1', 3.909017329083918)]

In [179]:
search("when did the Fed raise rates by 75 basis points")

6.74 2025-12-10_p5_c1 The Committee judges that reserve balances have declined to ample levels and will initiate purchases
6.63 2026-10-28_p5_c1 Voting against the monetary policy action were Lisa D. Cook and Beth M. Hammack, who preferred to ra
6.46 2025-08-22_p3_c1 The 2025 review included Fed Listens events at Reserve Banks around the country and a research confe
6.36 2026-07-29_p5_c1 Voting against the monetary policy action were Beth M. Hammack, Neel Kashkari, and Lorie K. Logan, w
5.81 2026-09-16_p2_c1 The Committee decided to raise the target range for the federal funds rate by 1/4 percentage point t


[('2025-12-10_p5_c1', 6.737073012930673),
 ('2026-10-28_p5_c1', 6.627103877192191),
 ('2025-08-22_p3_c1', 6.464621849539348),
 ('2026-07-29_p5_c1', 6.359099929811675),
 ('2026-09-16_p2_c1', 5.810786230397481)]